#Tiny GPT updated version

In [4]:

import torch
import torch.nn as nn
import torch.nn.functional as F

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ================== DATA ==================
corpus = [
    "hello friends how are you",
    "the tea is very hot",
    "my name is Adi",
    "the roads of Bangalore are busy always",
    "it is raining in Indranagar",
    "the train is late again",
    "i like having cars",
    "onam is my favorite festival",
    "diwali brings light and sweets",
    "india won cricket match",
    "bangalore has many it companies",
    "i love eating dosa and idli",
    "the weather is pleasant today",
    "cricket is very popular in india",
    "festival season brings joy and lights",
    "i live in bangalore karnataka",
]

corpus = [s.strip() + " <END>" for s in corpus]
text = "\n".join(corpus)

words = sorted(list(set(text.split())))
vocab_size = len(words)
print(f"Vocabulary size: {vocab_size}")

word2idx = {w: i for i, w in enumerate(words)}
idx2word = {i: w for w, i in word2idx.items()}

data = torch.tensor([word2idx[w] for w in text.split()], dtype=torch.long)

# ================== HYPERPARAMETERS ==================
block_size = 8
embedding_dim = 64
n_heads = 4
n_layers = 4
dropout = 0.1
batch_size = 16
lr = 3e-4
epochs = 3000          # you can increase later
eval_interval = 300

# Train / Val split
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split='train'):
    data_split = train_data if split == 'train' else val_data
    ix = torch.randint(len(data_split) - block_size, (batch_size,))
    x = torch.stack([data_split[i:i+block_size] for i in ix])
    y = torch.stack([data_split[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

# ================== TRANSFORMER BLOCKS (inside notebook) ==================
class SelfAttentionHead(nn.Module):
    def __init__(self, embedding_dim, block_size, head_size, dropout=0.1):
        super().__init__()
        self.key = nn.Linear(embedding_dim, head_size, bias=False)
        self.query = nn.Linear(embedding_dim, head_size, bias=False)
        self.value = nn.Linear(embedding_dim, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) / (C ** 0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        return wei @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, embedding_dim, block_size, num_heads, dropout=0.1):
        super().__init__()
        head_size = embedding_dim // num_heads
        self.heads = nn.ModuleList([SelfAttentionHead(embedding_dim, block_size, head_size, dropout) for _ in range(num_heads)])
        self.proj = nn.Linear(num_heads * head_size, embedding_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        return self.dropout(out)

class FeedForward(nn.Module):
    def __init__(self, n_embd, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, embedding_dim, block_size, n_heads, dropout=0.1):
        super().__init__()
        self.sa = MultiHeadAttention(embedding_dim, block_size, n_heads, dropout)
        self.ffwd = FeedForward(embedding_dim, dropout)
        self.ln1 = nn.LayerNorm(embedding_dim)
        self.ln2 = nn.LayerNorm(embedding_dim)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

# ================== MODEL ==================
class TinyGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, embedding_dim)
        self.position_embedding = nn.Embedding(block_size, embedding_dim)
        self.blocks = nn.Sequential(*[Block(embedding_dim, block_size, n_heads, dropout) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(embedding_dim)
        self.head = nn.Linear(embedding_dim, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding(idx)
        pos_emb = self.position_embedding(torch.arange(T, device=idx.device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.head(x)

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))
        return logits, loss

    def generate(self, idx, max_new_tokens=50, temperature=0.85, top_k=15):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')
            probs = F.softmax(logits, dim=-1)
            next_idx = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_idx), dim=1)
        return idx

# ================== TRAINING SETUP ==================
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = TinyGPT().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

print(f"Model has {sum(p.numel() for p in model.parameters()):,} parameters on {device}")

@torch.no_grad()
def estimate_loss():
    model.eval()
    losses = {'train': 0, 'val': 0}
    for split in ['train', 'val']:
        for _ in range(10):
            xb, yb = get_batch(split)
            _, loss = model(xb, yb)
            losses[split] += loss.item()
        losses[split] /= 10
    model.train()
    return losses

# ================== TRAINING LOOP ==================
for step in range(epochs):
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    if step % eval_interval == 0 or step == epochs-1:
        losses = estimate_loss()
        print(f"Step {step:4d} | Train Loss: {losses['train']:.4f} | Val Loss: {losses['val']:.4f}")

# ================== GENERATION ==================
print("\n" + "="*70)
print("GENERATED SENTENCES")
print("="*70)

start_words = ["hello", "raining", "bangalore", "india", "diwali", "onam","Adi","dosa"]

for word in start_words:
    if word in word2idx:
        context = torch.tensor([[word2idx[word]]], dtype=torch.long, device=device)
        out = model.generate(context, max_new_tokens=40, temperature=0.9, top_k=20)
        generated = " ".join(idx2word[int(i)] for i in out[0])
        print(f"Start with '{word}':")
        print(generated)
        print("-" * 60)

print("\nTraining finished! You can increase epochs for better results.")

Torch version: 2.10.0+cpu
CUDA available: False
Vocabulary size: 59
Model has 207,419 parameters on cpu
Step    0 | Train Loss: 4.3399 | Val Loss: 4.2155
Step  300 | Train Loss: 0.2660 | Val Loss: 4.3730
Step  600 | Train Loss: 0.1406 | Val Loss: 4.9301
Step  900 | Train Loss: 0.1145 | Val Loss: 5.3603
Step 1200 | Train Loss: 0.0959 | Val Loss: 5.3909
Step 1500 | Train Loss: 0.0936 | Val Loss: 5.5941
Step 1800 | Train Loss: 0.1139 | Val Loss: 5.7852
Step 2100 | Train Loss: 0.1059 | Val Loss: 5.9931
Step 2400 | Train Loss: 0.1047 | Val Loss: 6.0663
Step 2700 | Train Loss: 0.0819 | Val Loss: 6.2330
Step 2999 | Train Loss: 0.1026 | Val Loss: 6.2150

GENERATED SENTENCES
Start with 'hello':
hello friends how are you <END> the tea is very hot <END> my name is Adi <END> the roads of Bangalore are busy always <END> it is raining in Indranagar <END> the train is late again <END> i like having cars
------------------------------------------------------------
Start with 'raining':
raining in Indr